# `Genesis` demo — mobile dual-arm FR3 

This notebook shows how to use the reusable `RobotScene` class from
`scripts/scene_robot_tables.py`:

1. **Start** the simulation environment
2. **Drive the base** around the scene (forward / strafe / rotate)
3. **Control individual joints** (arms + grippers)
4. **Use IK** to move the left / right arm end-effectors

### 0. Install dependencies (AUPLC and Google Colab)

The cell below auto-detects the platform and installs only what's needed. It is a no-op
on Binder / a local install (dependencies already present).

> Google Colab: first set the runtime to GPU — **Runtime → Change runtime type → GPU**.

> On a fresh Colab / AUPLC session, **restart the kernel** after the install cell finishes.

In [1]:
# Colab Dependencies
# !apt update -qq && apt install -qq libnvidia-gl-580
# !pip install genesis-world==1.1.0 gs-nyx gs-nyx-plugin
# !pip install --no-deps git+https://github.com/yxzhan/fr3-genesis.git@nyx

# AUPLC Dependencies
# !pip install --no-deps git+https://github.com/yxzhan/fr3-genesis.git@main

## 1. Start the simulation

Toggle **`HEADLESS`** below:
* `HEADLESS = True` — no viewer window; you only see the inline frames (`show()`) and the
  final mp4. This is the right choice for a remote/browser kernel with no display.
* `HEADLESS = False` — opens the native Genesis viewer window (a separate desktop window;
  it does **not** appear inside the browser). Requires a display (`DISPLAY` set). The inline
  `show()` frames and the mp4 still work in addition to the window.

In [ ]:
import os
import numpy as np
import cv2
import ipywidgets as widgets
from IPython.display import display

# fr3_genesis is installed as a package (uv sync), so import RobotScene directly.
from fr3_genesis import RobotScene

# Toggle this: True = no viewer window (offscreen only),
#              False = also open the native Genesis viewer (needs a display).
HEADLESS = True
SAVE_VIDEO = True

if not HEADLESS:
    SAVE_VIDEO = False
    from utils import display_desktop
    display_desktop()

# high_quality=True -> ray-traced main camera (needs the gs-nyx plugin, slower).
sim = RobotScene(headless=HEADLESS)

if SAVE_VIDEO:
    # The output path is fixed here, at start_recording(); save_video() takes no path.
    VIDEO_PATH = os.path.join(os.path.abspath("."), "videos", "robot_scene_tables.mp4")
    sim.start_recording(VIDEO_PATH)   # begin accumulating frames for the final mp4

[I 06/04/26 21:32:17.495 14165] [shell.py:_shell_pop_print@25] Graphical python shell detected, using wrapped sys.stdout


[backend] using: cuda (precision=32)
[Genesis] [21:32:19] [INFO] ╭───────────────────────────────────────────────╮
[Genesis] [21:32:19] [INFO] │┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈ Genesis ┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈│
[Genesis] [21:32:19] [INFO] ╰───────────────────────────────────────────────╯
[Genesis] [21:32:20] [INFO] Running on [NVIDIA GeForce RTX 2070] with backend gs.cuda. Device memory: 7.60 GB.
[Genesis] [21:32:20] [INFO] 🚀 Genesis initialized. 🔖 version: 1.1.0, 🎨 theme: light, 🌱 seed: None, 🐛 debug: False, 📏 precision: 32, 🔥 performance: False, 💬 verbose: INFO
[2026-06-04 21:32:30.896] [CoACD] [info] threshold               0.1
[2026-06-04 21:32:30.896] [CoACD] [info] max # convex hull       -1
[2026-06-04 21:32:30.896] [CoACD] [info] preprocess mode         auto
[2026-06-04 21:32:30.896] [CoACD] [info] preprocess resolution   30
[2026-06-04 21:32:30.896] [CoACD] [info] pca                     false
[2026-06-04 21:32:30.896] [CoACD] [info] mcts max depth          3
[2026-06-04 21:32:30.896] [CoACD] [

[Genesis] [21:32:58] [WARNING] Link 'imagetostl_mesh_0_009' has dubious mass 138.425 compared to the estimate from geometry 0.091 given material density 600.
[Genesis] [21:32:58] [WARNING] Link 'imagetostl_mesh_0_011' has dubious mass 15.822 compared to the estimate from geometry 0.017 given material density 600.
[Genesis] [21:32:59] [WARNING] Ignoring inertia matrix of link 'argo_drive_front_link' because center of mass is not specified.
[Genesis] [21:32:59] [WARNING] Ignoring inertia matrix of link 'argo_drive_rear_link' because center of mass is not specified.
[Genesis] [21:33:00] [WARNING] Neutral robot position (qpos0) exceeds joint limits.
[Genesis] [21:33:18] [WARNING] Filtered out geometry pairs causing self-collision for the neutral configuration (qpos0): (93, 104), (93, 105), (93, 108), (93, 109), (95, 105), (97, 109), (116, 120), (116, 121), (116, 128), (117, 122), (117, 123), (117, 136), (127, 131), (135, 139). Consider tuning Morph option 'decompose_robot_error_threshold' 

In [ ]:
import matplotlib.pyplot as plt

def show(img, title=None):
    """Render one frame from the offscreen camera and display it inline."""
    plt.figure(figsize=(8, 5))
    plt.imshow(img)
    plt.axis("off")
    if title:
        plt.title(title)
    plt.show()

## 2. Drive the base

`sim.set_base_velocity(vx, vy, wz, steps)` sets a body-frame velocity and advances the simulation:
* `vx` — forward (+) / back (−), m/s
* `vy` — strafe left (+) / right (−), m/s
* `wz` — rotate in place, left (+) / right (−), rad/s

The dual-steer wheels and a heading-hold controller are handled internally.

In [ ]:
print("start base pose:", np.round(sim.get_base_pose()[0], 3))

sim.set_base_velocity(0.0, 0.0, -1.2)   # rotate in place (left)
sim.step(200)
# show(sim.render_head(), "after rotating")
show(sim.render(), "after rotating")


sim.set_base_velocity(0.5, 0.0, 0.0)    # forward
sim.step(200)
show(sim.render(), "after driving forward")


sim.set_base_velocity(0.0, 0.5, 0.0)    # strafe left
sim.step(200)
show(sim.render(), "after strafing left")

sim.stop_base()
pos, yaw = sim.get_base_pose()
print("end base pose:", np.round(pos, 3), "yaw(deg):", round(np.degrees(yaw), 1))

## 3. Control individual joints

Each arm has 7 joints and a 2-finger gripper. Set position targets directly:
* `sim.set_arm(side, q)` — 7 joint angles (rad) for `side` in `'left'` / `'right'`
* `sim.set_gripper(side, opening)` — finger opening in meters (`0.0` closed .. `0.04` open)

Targets are held every simulation step; pass `step=True, steps=N` to advance immediately.

In [ ]:
# Read the current left-arm joint angles, then bend joint 4 (elbow) and joint 6 (wrist).
q_left = sim.get_arm("left").copy()
print("left arm q (rad):", np.round(q_left, 2))

q_left[3] += 0.6    # joint4
q_left[5] -= 0.5    # joint6
sim.set_arm("left", q_left)
sim.step(120)

show(sim.render(), "after left arm joint control")

# Close then open the left gripper.
sim.set_gripper("left", 0.0)     # close
sim.step(50)
sim.set_gripper("left", 0.04)    # open again
sim.step(50)

# Torso Lift
sim.set_spine(0.8)
sim.step(100)


## 4. Pick and place a cube with IK

A small green cube sits on the cutlery table. We:

1. **Teleport** the base next to the table with `sim.teleport_base(x, y, yaw)` so the
   right arm can reach the cube (instant, no driving).
2. Read the cube's live pose with `sim.cube.get_pos()`.
3. Run a **right-arm pick-and-place** using the IK move primitive `sim.move_ee(side, pos)`
   (moves the finger TCP along a straight line) plus `sim.set_gripper(side, opening)`:
   open → above cube → down → close → lift → over a floor spot → down → open (drop).

The numeric poses below (teleport target, approach/drop heights) are good starting
points; tweak them if the arm doesn't quite reach or the grasp slips.

In [ ]:
# --- 0. reset the arms to the tucked hold pose (undo the earlier joint/IK moves) ---
sim.scene.reset()
sim.reset_pose()
sim.step(100)

# --- 1. teleport to the OPEN FRONT (+y) side of the cutlery table, facing it (-y) ---
# Approaching from +x would drive the wide base into the table; the +y side is open
# (next table is 1.5 m away), so the body clears while the arm can still reach over.
# yaw=-pi/2 -> body +x points to world -y (toward the table); head cam looks at it.
sim.teleport_base(-1.8, 4, yaw=-np.pi / 2)
show(sim.render_head(), "teleported in front of the table")

# --- 2. read the cube's live pose (it has settled onto the table) ---
cube = sim.cube.get_pos().cpu().numpy()
print("cube at:", np.round(cube, 3))

side = "right"
APPROACH = np.array([0.0, 0.0, 0.15])   # 15 cm above a target before descending
# Don't drive the TCP all the way to the cube centre, or the fingertips reach the
# tabletop. Grip the upper part of the 4 cm cube: descend to centre + ~2 cm so the
# fingertips stay clear of the table.
GRASP = np.array([0.0, 0.0, 0.05])      # grasp-height clearance above the cube centre
# DROP_SPOT = np.array([-1.82, 3.55, 0.12])  # drop spot: open floor in front of the table
# DROP_SPOT = np.array([-2.12, 3, 0.85])  # drop spot: Plate
DROP_SPOT = np.array([-1.88, 3, 0.92])  # drop spot: Bowl



# --- 3. right-arm pick-and-place ---
sim.set_spine(0.3)
sim.set_gripper(side, 0.04)             # open
sim.step(50)
print("right TCP start:", np.round(sim.get_ee_pos(side), 3))

sim.move_ee(side, tuple(cube + APPROACH))   # hover above the cube
sim.step(100)
show(sim.render_head(), "hover above the cube")

sim.move_ee(side, tuple(cube + GRASP))      # descend to grasp height (above the tabletop)
sim.step(100)
show(sim.render_head(), "descend to the cube")

sim.set_gripper(side, 0.0)                  # close on the cube
sim.step(50)
show(sim.render_head(), "grasped the cube")

sim.move_ee(side, tuple(cube + APPROACH))   # lift straight up
sim.step(100)
show(sim.render_head(), "lift straight up")

sim.move_ee(side, tuple(DROP_SPOT + APPROACH)) # travel over the drop spot
sim.step(100)
show(sim.render_head(), "travel over the drop spot")

sim.move_ee(side, tuple(DROP_SPOT))            # lower to the floor
sim.step(100)
show(sim.render_head(), "lower to the drop spot")

sim.set_gripper(side, 0.04)                 # release
sim.step(50)
show(sim.render_head(), "placed the cube")

sim.move_ee(side, tuple(DROP_SPOT + APPROACH))   # hover above
sim.step(100)
show(sim.render_head(),"hover above")

print("cube final pos:", np.round(sim.cube.get_pos().cpu().numpy(), 3))

## 5. Save and embed the recorded video

Everything above was recorded (we called `start_recording()` in step 1). Write it to an
mp4 and embed it inline — the second way to visualize the run.

In [ ]:
from IPython.display import Video

# Output path was set in step 1 via start_recording(); save_video() takes no path.
result = sim.save_video()
if result is not None:
    top_path, head_path = result
    display(Video(top_path, embed=True, width=720))
    display(Video(head_path, embed=True, width=720))
else:
    print("No video to display (set SAVE_VIDEO = True in the first cell to record).")